# Fine-tuning walkthrough

This notebook walks through the same pipeline as the scripts: prepare data, train with LoRA, evaluate, and generate samples. It mirrors `scripts/prepare_data.py` and `scripts/run_train.py` cell-for-cell, so the analysis is reproducible from the command line too.

Run order: `scripts/prepare_data.py` → `scripts/run_train.py configs/default.yaml` → `scripts/run_evaluate.py configs/default.yaml`.

## 1. Data preparation

Sample `databricks/databricks-dolly-15k` (CC-BY-SA-3.0), format into the instruction template, split deterministically.

In [ ]:
from llm_finetune.data_prep import build_examples, split_examples
from datasets import load_dataset

ds = load_dataset("databricks/databricks-dolly-15k", split="train").shuffle(seed=42).select(range(1000))
rows = [{"instruction": r["instruction"], "context": r["context"], "response": r["response"]} for r in ds]
examples = build_examples(rows)
train, val = split_examples(examples, val_fraction=0.1, seed=42)
print(f"{len(train)} train / {len(val)} val examples")
print(examples[0][:200])

## 2. Training

LoRA fine-tune `distilgpt2` with the config-driven Trainer.

In [ ]:
from llm_finetune.config import load_config
from llm_finetune.train import train

cfg = load_config("configs/default.yaml")
metrics = train(cfg)
print(f"baseline perplexity: {metrics['baseline_perplexity']:.2f}")
print(f"final perplexity:    {metrics['final_perplexity']:.2f}")

## 3. Evaluation + samples

Generate responses to held-out instructions.

In [ ]:
from llm_finetune.inference import generate, instruction_prompt, load_adapter

model, tokenizer = load_adapter(cfg.model_name, cfg.output_dir)
for instruction in ["What is a database index?", "Write a short thank-you note"]:
    out = generate(model, tokenizer, instruction_prompt(instruction))
    print(f"### {instruction}\n{out}\n")